In [1]:
# Unzip the diabetes-data.tar.Z file

import io
import tarfile
from pathlib import Path
import unlzw3

z_path = Path("../assets/dataset/raw/diabetes-data.tar.Z").resolve()
out_dir = z_path.parent

blob = z_path.read_bytes()
decompressed = unlzw3.unlzw(blob)

# PyPI readme says string; tar data must be bytes for tarfile.
if isinstance(decompressed, str):
    decompressed = decompressed.encode("latin-1")

with tarfile.open(fileobj=io.BytesIO(decompressed), mode="r:") as tf:
    tf.extractall(path=out_dir)

In [5]:
# Format txt into CSV format
import re
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../assets/dataset/Diabetes-Data").resolve()  # adjust if cwd differs
OUTPUT = DATA_DIR / "diabetes_all_patients.csv"

pattern = re.compile(r"^data-(\d+)$")

chunks = []
for path in sorted(DATA_DIR.iterdir()):
    m = pattern.match(path.name)
    if not m or not path.is_file():
        continue
    patient_id = int(m.group(1))

    df = pd.read_csv(
        path,
        sep="\t",
        header=None,
        names=["Date", "Time", "Code", "Value"],
        dtype=str,  # keeps leading zeros in Value (e.g. "009")
    )
    df.insert(0, "patientId", patient_id)
    chunks.append(df)

result = pd.concat(chunks, ignore_index=True)
result.to_csv(OUTPUT, index=False)
print(len(chunks), "patients,", len(result), "rows ->", OUTPUT)

70 patients, 29330 rows → C:\Users\ahmad\OneDrive\Desktop\SchoolProjects\ModelingOptimization\AppliedModelingOptimization\Ahmad\assets\dataset\Diabetes-Data\diabetes_all_patients.csv


In [6]:
# Format txt into CSV format with codes
import re
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../assets/dataset/Diabetes-Data").resolve()  # adjust if cwd differs
OUTPUT = DATA_DIR / "diabetes_all_patients_with_codes.csv"

CODE_MEANINGS = {
    "33": "Regular insulin dose",
    "34": "NPH insulin dose",
    "35": "UltraLente insulin dose",
    "48": "Unspecified blood glucose measurement",
    "57": "Unspecified blood glucose measurement",
    "58": "Pre-breakfast blood glucose measurement",
    "59": "Post-breakfast blood glucose measurement",
    "60": "Pre-lunch blood glucose measurement",
    "61": "Post-lunch blood glucose measurement",
    "62": "Pre-supper blood glucose measurement",
    "63": "Post-supper blood glucose measurement",
    "64": "Pre-snack blood glucose measurement",
    "65": "Hypoglycemic symptoms",
    "66": "Typical meal ingestion",
    "67": "More-than-usual meal ingestion",
    "68": "Less-than-usual meal ingestion",
    "69": "Typical exercise activity",
    "70": "More-than-usual exercise activity",
    "71": "Less-than-usual exercise activity",
    "72": "Unspecified special event",
}

pattern = re.compile(r"^data-(\d+)$")

chunks = []
for path in sorted(DATA_DIR.iterdir()):
    m = pattern.match(path.name)
    if not m or not path.is_file():
        continue

    patient_id = int(m.group(1))

    df = pd.read_csv(
        path,
        sep="\t",
        header=None,
        names=["Date", "Time", "Code", "Value"],
        dtype=str,  # keeps leading zeros in Value (e.g. "009")
    )

    df.insert(0, "patientId", patient_id)
    df["CodeMeaning"] = df["Code"].map(CODE_MEANINGS).fillna("Unknown code")
    chunks.append(df)

result = pd.concat(chunks, ignore_index=True)
result.to_csv(OUTPUT, index=False)

print(len(chunks), "patients,", len(result), "rows ->", OUTPUT)


70 patients, 29330 rows -> C:\Users\ahmad\OneDrive\Desktop\SchoolProjects\ModelingOptimization\AppliedModelingOptimization\Ahmad\assets\dataset\Diabetes-Data\diabetes_all_patients_with_codes.csv


In [9]:
import re
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../assets/dataset/Diabetes-Data").resolve()
OUTPUT = DATA_DIR / "diabetes_all_patients.csv"

CODE_MEANINGS = {
    "33": "Regular insulin dose",
    "34": "NPH insulin dose",
    "35": "UltraLente insulin dose",
    "48": "Unspecified blood glucose measurement",
    "57": "Unspecified blood glucose measurement",
    "58": "Pre-breakfast blood glucose measurement",
    "59": "Post-breakfast blood glucose measurement",
    "60": "Pre-lunch blood glucose measurement",
    "61": "Post-lunch blood glucose measurement",
    "62": "Pre-supper blood glucose measurement",
    "63": "Post-supper blood glucose measurement",
    "64": "Pre-snack blood glucose measurement",
    "65": "Hypoglycemic symptoms",
    "66": "Typical meal ingestion",
    "67": "More-than-usual meal ingestion",
    "68": "Less-than-usual meal ingestion",
    "69": "Typical exercise activity",
    "70": "More-than-usual exercise activity",
    "71": "Less-than-usual exercise activity",
    "72": "Unspecified special event",
}

pattern = re.compile(r"^data-(\d+)$")

chunks = []
for path in sorted(DATA_DIR.iterdir()):
    m = pattern.match(path.name)
    if not m or not path.is_file():
        continue

    patient_id = int(m.group(1))

    df = pd.read_csv(
        path,
        sep="\t",
        header=None,
        names=["Date", "Time", "Code", "Value"],
        dtype=str,
    )

    df.insert(0, "patientId", patient_id)

    # Assign sequential day numbers by unique date order within each patient file
    df["Day"] = pd.factorize(df["Date"])[0] + 1
    time_parsed = pd.to_datetime(df["Time"], format="%H:%M", errors="coerce")
    df["MinuteOfDay"] = time_parsed.dt.hour * 60 + time_parsed.dt.minute


    df["CodeMeaning"] = df["Code"].map(CODE_MEANINGS).fillna("Unknown code")

    df = df.drop(columns=["Date", "Time"])
    df = df[["patientId", "Day", "MinuteOfDay", "Code", "Value", "CodeMeaning"]]

    chunks.append(df)

result = pd.concat(chunks, ignore_index=True)
result.to_csv(OUTPUT, index=False)

print(len(chunks), "patients,", len(result), "rows ->", OUTPUT)


70 patients, 29330 rows -> C:\Users\ahmad\OneDrive\Desktop\SchoolProjects\ModelingOptimization\AppliedModelingOptimization\Ahmad\assets\dataset\Diabetes-Data\diabetes_all_patients.csv


In [4]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

DATA_PATH = Path("../assets/dataset/CSV/diabetes_all_patients_code.csv")

df = pd.read_csv(DATA_PATH, sep=None, engine="python")

# Basic cleanup
df["patientId"] = pd.to_numeric(df["patientId"], errors="coerce")
df["Day"] = pd.to_numeric(df["Day"], errors="coerce")
df["MinuteOfDay"] = pd.to_numeric(df["MinuteOfDay"], errors="coerce")
df["Code"] = df["Code"].astype(str)
df["ValueNum"] = pd.to_numeric(df["Value"], errors="coerce")

glucose_codes = {"48", "57", "58", "59", "60", "61", "62", "63", "64"}
insulin_codes = {"33", "34", "35"}

code_colors = {
    "48": "gray",
    "57": "black",
    "58": "blue",
    "59": "cyan",
    "60": "green",
    "61": "lime",
    "62": "orange",
    "63": "red",
    "64": "purple",
}

insulin_colors = {
    "33": "magenta",      # Regular
    "34": "brown",        # NPH
    "35": "darkgoldenrod" # UltraLente
}

def minute_to_hhmm(m):
    h = int(m // 60)
    mm = int(m % 60)
    return f"{h:02d}:{mm:02d}"

def get_days_for_patient(patient_id):
    days = (
        df.loc[df["patientId"] == patient_id, "Day"]
        .dropna()
        .sort_values()
        .unique()
        .tolist()
    )
    return [int(d) for d in days]

def plot_patient_day(patient_id, day, show_labels=True):
    day_data = df[
        (df["patientId"] == patient_id) &
        (df["Day"] == day)
    ].copy()

    day_data = day_data.dropna(subset=["MinuteOfDay"])
    day_data = day_data.sort_values("MinuteOfDay")

    glucose_data = day_data[
        day_data["Code"].isin(glucose_codes) & day_data["ValueNum"].notna()
    ].copy()

    insulin_data = day_data[
        day_data["Code"].isin(insulin_codes) & day_data["ValueNum"].notna()
    ].copy()

    event_codes = {"65", "66", "67", "68", "69", "70", "71", "72"}
    event_data = day_data[day_data["Code"].isin(event_codes)].copy()

    if day_data.empty:
        print(f"No data found for patient {patient_id}, day {day}.")
        return

    fig, ax1 = plt.subplots(figsize=(14, 7))
    ax2 = ax1.twinx()

    # Glucose colors
    glucose_colors = {
        "48": "gray",
        "57": "black",
        "58": "blue",
        "59": "deepskyblue",
        "60": "green",
        "61": "limegreen",
        "62": "orange",
        "63": "red",
        "64": "purple",
    }

    # Insulin colors
    insulin_colors = {
        "33": "magenta",
        "34": "brown",
        "35": "darkgoldenrod",
    }

    # Timeline event colors
    event_colors = {
        "65": "red",        # Hypoglycemic symptoms
        "66": "teal",       # Typical meal
        "67": "darkgreen",  # More-than-usual meal
        "68": "lightgreen", # Less-than-usual meal
        "69": "navy",       # Typical exercise
        "70": "darkblue",   # More-than-usual exercise
        "71": "skyblue",    # Less-than-usual exercise
        "72": "black",      # Special event
    }

    # Plot glucose measurements
    for code, group in glucose_data.groupby("Code"):
        color = glucose_colors.get(code, "gray")
        label = f"{code}: {group['CodeMeaning'].iloc[0]}"

        ax1.scatter(
            group["MinuteOfDay"],
            group["ValueNum"],
            color=color,
            s=90,
            label=label,
            zorder=3
        )

        ax1.plot(
            group["MinuteOfDay"],
            group["ValueNum"],
            color=color,
            alpha=0.35,
            zorder=2
        )

        if show_labels:
            for _, row in group.iterrows():
                ax1.text(
                    row["MinuteOfDay"],
                    row["ValueNum"] + 4,
                    f"{int(row['ValueNum'])}",
                    color=color,
                    fontsize=8,
                    ha="center"
                )

    # Plot insulin doses
    for code, group in insulin_data.groupby("Code"):
        color = insulin_colors.get(code, "black")
        label = f"{code}: {group['CodeMeaning'].iloc[0]}"

        ax2.scatter(
            group["MinuteOfDay"],
            group["ValueNum"],
            color=color,
            marker="^",
            s=110,
            label=label,
            zorder=4
        )

        if show_labels:
            for _, row in group.iterrows():
                ax2.text(
                    row["MinuteOfDay"],
                    row["ValueNum"] + 0.3,
                    f"{row['ValueNum']:.0f}",
                    color=color,
                    fontsize=8,
                    ha="center"
                )

    # Plot non-glucose / non-insulin events as vertical lines
    for code, group in event_data.groupby("Code"):
        color = event_colors.get(code, "black")
        label = f"{code}: {group['CodeMeaning'].iloc[0]}"

        for _, row in group.iterrows():
            ax1.axvline(
                x=row["MinuteOfDay"],
                color=color,
                linestyle="--",
                alpha=0.45,
                linewidth=1
            )

        # Add one invisible point so the legend includes this event type
        ax1.scatter([], [], color=color, marker="|", s=200, label=label)

        if show_labels:
            for _, row in group.iterrows():
                ax1.text(
                    row["MinuteOfDay"],
                    ax1.get_ylim()[1] * 0.98,
                    code,
                    color=color,
                    fontsize=8,
                    rotation=90,
                    va="top",
                    ha="center"
                )

    ax1.set_title(f"Patient {patient_id} - Day {day}")
    ax1.set_xlabel("Time of Day")
    ax1.set_ylabel("Glucose Value")
    ax2.set_ylabel("Insulin Dose")

    xticks = list(range(0, 1441, 120))
    ax1.set_xticks(xticks)
    ax1.set_xticklabels([minute_to_hhmm(x) for x in xticks], rotation=45)

    ax1.grid(True, alpha=0.3)

    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax1.legend(h1 + h2, l1 + l2, bbox_to_anchor=(1.02, 1), loc="upper left")

    plt.tight_layout()
    plt.show()



In [5]:
patient_options = sorted(df["patientId"].dropna().astype(int).unique().tolist())

patient_widget = widgets.Dropdown(
    options=patient_options,
    value=patient_options[0],
    description="Patient:"
)

day_widget = widgets.Dropdown(
    options=get_days_for_patient(patient_options[0]),
    description="Day:"
)

label_widget = widgets.Checkbox(
    value=True,
    description="Show insulin dose labels"
)

def update_days(change):
    patient_id = change["new"]
    days = get_days_for_patient(patient_id)
    day_widget.options = days
    if days:
        day_widget.value = days[0]

patient_widget.observe(update_days, names="value")

ui = widgets.VBox([patient_widget, day_widget, label_widget])

out = widgets.interactive_output(
    plot_patient_day,
    {
        "patient_id": patient_widget,
        "day": day_widget,
        "show_labels": label_widget
    }
)

display(ui, out)


Output()

In [ ]:
def count_codes(dataframe):
    counts = (
        dataframe["Code"]
        .astype(str)
        .value_counts()
        .sort_index()
        .rename_axis("Code")
        .reset_index(name="Count")
    )

    meanings = (
        dataframe[["Code", "CodeMeaning"]]
        .dropna()
        .drop_duplicates()
        .copy()
    )
    meanings["Code"] = meanings["Code"].astype(str)

    counts = counts.merge(meanings, on="Code", how="left")
    return counts[["Code", "CodeMeaning", "Count"]]

print(count_codes(df))
'''
count_codes(df[df["patientId"] == 1])
count_codes(df[(df["patientId"] == 1) & (df["Day"] == 1)])
'''

   Code                               CodeMeaning  Count
0     0                              Unknown code     33
1    33                      Regular insulin dose   9518
2    34                          NPH insulin dose   3830
3    35                   UltraLente insulin dose   1053
4    36                              Unknown code      1
5     4                              Unknown code      1
6    48     Unspecified blood glucose measurement   1883
7    56                              Unknown code    119
8    57     Unspecified blood glucose measurement    990
9    58   Pre-breakfast blood glucose measurement   3518
10   59  Post-breakfast blood glucose measurement     20
11   60       Pre-lunch blood glucose measurement   2771
12   61      Post-lunch blood glucose measurement     66
13   62      Pre-supper blood glucose measurement   3160
14   63     Post-supper blood glucose measurement    219
15   64       Pre-snack blood glucose measurement    904
16   65                     Hyp

'\ncount_codes(df[df["patientId"] == 1])\ncount_codes(df[(df["patientId"] == 1) & (df["Day"] == 1)])\n'

: 